In [0]:
from pyspark.sql.functions import col, to_date, to_timestamp, hour, when, current_timestamp

In [0]:
bronze_stream = spark.readStream.table("dev.iot_fleet.bronze_telemetry")

In [0]:
silver_stream = bronze_stream \
    .withColumn("event_timestamp", to_timestamp("event_time")) \
    .withColumn("event_date", to_date("event_timestamp")) \
    .withColumn("event_hout", hour("event_timestamp")) \
    .withColumn("high_temp_flag", when(col("engine_temperature") > 105, 1).otherwise(0)) \
    .withColumn("low_fuel_flag", when(col("fuel_level") < 15, 1).otherwise(0)) \
    .withColumn("bad_sensor_flag", when(col("sensor_status") != "OK", 1).otherwise(0)) \
    .filter(col("event_id").isNotNull()) \
    .filter(col("vehicle_id").isNotNull()) \
    .filter(col("event_timestamp").isNotNull()) \
    .filter(col("latitude").between(-90, 90)) \
    .filter(col("longitude").between(-180, 180)) \
    .filter(col("speed").between(0, 160)) \
    .withColumn("silver_processed_timestamp", current_timestamp())

In [0]:
silver_cheakpoint = "/Volumes/dev/iot_fleet/checkpoints/silver"
silver_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", silver_cheakpoint) \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .toTable("dev.iot_fleet.silver_telemetry")

In [0]:
display(spark.table("dev.iot_fleet.silver_telemetry").limit(5))

In [0]:
spark.table("dev.iot_fleet.silver_telemetry").printSchema()

In [0]:
spark.sql("""
    SELECT count(*)
    FROM dev.iot_fleet.silver_telemetry
    WHERE speed < 0
    OR speed > 160
""").show()

In [0]:
spark.sql("""
    SELECT
        high_temp_flag,
        COUNT(*) AS records
    FROM dev.iot_fleet.silver_telemetry
    GROUP BY high_temp_flag
""").show()